# Prepare Dataset for Hybrid IDS

This notebook merges selected subsets of the **CICIDS2017** dataset into a single CSV that matches the format expected by the agentic IDS system.  

The resulting dataset uses the following columns:

- **`label`**: the attack type or `BENIGN` for normal traffic.
- **`payload`**: a placeholder string describing the flow.  
  The original CICIDS2017 flows do **not** include raw packet payloads, so this field is left empty or filled with a simple description.
- **`source_ip`**: IP address of the source host.
- **`destination_ip`**: IP address of the destination host.
- **`protocol`**: network protocol (e.g. `TCP`, `UDP`, `ICMP`).
- **`port`**: destination port.

The notebook extracts the dataset archive, reads the CSV files corresponding to DDoS, PortScan, Web Attack and Benign traffic, standardises column names, and concatenates them into a single DataFrame.

In [ ]:
import os
import zipfile
import pandas as pd

# Path to the CICIDS2017 archive
zip_path = "/content/Dataset 1-20260602T062444Z-3-001.zip"

# Directory to extract the CSV files
extract_dir = "Dataset 1"

# Extract the archive if necessary
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

# List of CSV files to include in the merged dataset
selected_files = [
    "Dataset 1/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Dataset 1/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Dataset 1/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
]

frames = []
for fname in selected_files:
    file_path = os.path.join(extract_dir, fname)
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found.")
        continue

    df = pd.read_csv(file_path)

    # Strip whitespace from column names to prevent KeyError
    df.columns = df.columns.str.strip()

    # Sample equally from each class (Label)
    n_samples_per_class = 250
    df = df.groupby('Label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), n_samples_per_class), random_state=42)).reset_index(drop=True)

    # Standardise column names to match the expected format
    df_standard = pd.DataFrame({
        "label": df["Label"],
        # CICIDS2017 does not include raw payloads; leave this empty or add a description
        "payload": ["" for _ in range(len(df))],
        "source_ip": df.get("Source IP", "0.0.0.0"),
        "destination_ip": df.get("Destination IP", "0.0.0.0"),
        "protocol": df.get("Protocol", "Unknown"),
        # Choose the destination port; depending on your use case you could pick the source port
        "port": df.get("Destination Port", 0),
    })
    frames.append(df_standard)

# Concatenate all selected files
if len(frames) > 0:
    merged_df = pd.concat(frames, ignore_index=True)

    # Save the merged dataset
    output_path = "cicids2017_merged.csv"
    merged_df.to_csv(output_path, index=False)
    print(f"Merged dataset saved to {output_path}.")
else:
    print("Error: No files were found to merge. Please check the filenames and extraction directory.")


/tmp/ipykernel_530/1117423586.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('Label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), n_samples_per_class), random_state=42)).reset_index(drop=True)
/tmp/ipykernel_530/1117423586.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('Label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), n_samples_per_cla

Merged dataset saved to cicids2017_merged.csv.


/tmp/ipykernel_530/1117423586.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('Label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), n_samples_per_class), random_state=42)).reset_index(drop=True)
